In [ ]:
# --- Baseline Notebook for Audio Emotion Recognition ---
# Competitors should improve this code with better preprocessing, architectures, and training strategies.

import os
import numpy as np
import librosa
import librosa.display
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

In [ ]:
# --- Dataset Loader ---
class AudioDataset(Dataset):
    def __init__(self, file_paths, labels, sr=22050, duration=3):
        self.file_paths = file_paths
        self.labels = labels
        self.sr = sr
        self.duration = duration

    def __len__(self):
        return len(self.file_paths)

    def __getitem__(self, idx):
        file_path = self.file_paths[idx]
        label = self.labels[idx]

        # Load audio
        y, sr = librosa.load(file_path, sr=self.sr, duration=self.duration)
        # Convert to Mel-spectrogram
        mel_spec = librosa.feature.melspectrogram(y=y, sr=sr)
        mel_spec_db = librosa.power_to_db(mel_spec, ref=np.max)

        # Normalize
        mel_spec_db = (mel_spec_db - mel_spec_db.mean()) / mel_spec_db.std()

        # Convert to tensor
        mel_tensor = torch.tensor(mel_spec_db).unsqueeze(0).float()
        return mel_tensor, torch.tensor(label).long()

# --- Simple CNN Model ---
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=8):
        super(SimpleCNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, stride=1, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, stride=1, padding=1)
        self.fc1 = nn.Linear(32 * 32 * 32, 128)  # adjust depending on input size
        self.fc2 = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.pool(torch.relu(self.conv1(x)))
        x = self.pool(torch.relu(self.conv2(x)))
        x = x.view(x.size(0), -1)
        x = torch.relu(self.fc1(x))
        x = self.fc2(x)
        return x

# --- Training Loop (simplified) ---
def train(model, dataloader, criterion, optimizer, epochs=5):
    for epoch in range(epochs):
        running_loss = 0.0
        for inputs, labels in dataloader:
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
        print(f"Epoch {epoch+1}, Loss: {running_loss/len(dataloader)}")

In [ ]:

# --- Example Usage ---
# file_paths = [...]  # list of audio file paths
# labels = [...]      # corresponding emotion labels (0–7)
# dataset = AudioDataset(file_paths, labels)
# dataloader = DataLoader(dataset, batch_size=16, shuffle=True)

# model = SimpleCNN(num_classes=8)
# criterion = nn.CrossEntropyLoss()
# optimizer = optim.Adam(model.parameters(), lr=0.001)

# train(model, dataloader, criterion, optimizer, epochs=5)